# Deep linear network

A deep linear network with one neuron per layer: every layer maps a single scalar input to a single scalar output, with no nonlinear activation.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch import nn
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots


class OneNeuron(nn.Module):
    def __init__(self, depth: int):
        super().__init__()

        self.layers = nn.Sequential(
            *[nn.Linear(1, 1, bias=False) for _ in range(depth)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


In [2]:
depth = 8
model = OneNeuron(depth)
model


OneNeuron(
  (layers): Sequential(
    (0): Linear(in_features=1, out_features=1, bias=False)
    (1): Linear(in_features=1, out_features=1, bias=False)
    (2): Linear(in_features=1, out_features=1, bias=False)
    (3): Linear(in_features=1, out_features=1, bias=False)
    (4): Linear(in_features=1, out_features=1, bias=False)
    (5): Linear(in_features=1, out_features=1, bias=False)
    (6): Linear(in_features=1, out_features=1, bias=False)
    (7): Linear(in_features=1, out_features=1, bias=False)
  )
)

In [3]:
x = torch.linspace(-1.0, 1.0, steps=5).unsqueeze(1)
y = model(x)

print(x.shape, "->", y.shape)
print(y.detach().flatten())


torch.Size([5, 1]) -> torch.Size([5, 1])
tensor([-0.0004, -0.0002,  0.0000,  0.0002,  0.0004])


## Train loss, sharpness, and optimizers

In [7]:
seed = 0
depth = 8
x_train = torch.linspace(-1.0, 1.0, steps=64).unsqueeze(1)
y_train = 2.0 * x_train
loss_fn = nn.MSELoss()

iterations = 3000
base_lr = 0.1
momentum_lr = 0.02
adam_lr = 0.01
batch_size = 16

optimizer_specs = [
    {
        "name": "GD",
        "batch_mode": "full",
        "lr": base_lr,
        "factory": lambda parameters: torch.optim.SGD(parameters, lr=base_lr),
    },
    {
        "name": "SGD",
        "batch_mode": "mini_batch",
        "lr": base_lr,
        "factory": lambda parameters: torch.optim.SGD(parameters, lr=base_lr),
    },
    {
        "name": "SGD + Polyak momentum",
        "batch_mode": "mini_batch",
        "lr": momentum_lr,
        "factory": lambda parameters: torch.optim.SGD(
            parameters, lr=momentum_lr, momentum=0.9
        ),
    },
    {
        "name": "Adam",
        "batch_mode": "mini_batch",
        "lr": adam_lr,
        "factory": lambda parameters: torch.optim.Adam(parameters, lr=adam_lr),
    },
]

In [5]:
def weights_vector(model: nn.Module) -> torch.Tensor:
    return torch.stack([layer.weight.reshape(()) for layer in model.layers])


def loss_from_weights(weights: torch.Tensor, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return ((x * weights.prod() - target) ** 2).mean()


def sharpness(model: nn.Module, x: torch.Tensor, target: torch.Tensor) -> float:
    weights = weights_vector(model).detach().requires_grad_(True)
    hessian = torch.autograd.functional.hessian(
        lambda current_weights: loss_from_weights(current_weights, x, target),
        weights,
    )
    return torch.linalg.eigvalsh(hessian).max().item()


def select_batch(batch_mode: str, generator: torch.Generator) -> tuple[torch.Tensor, torch.Tensor]:
    if batch_mode == "full":
        return x_train, y_train

    indices = torch.randperm(x_train.shape[0], generator=generator)[:batch_size]
    return x_train[indices], y_train[indices]


def run_experiment(optimizer_spec: dict) -> dict:
    torch.manual_seed(seed)
    batch_generator = torch.Generator().manual_seed(seed)

    model = OneNeuron(depth)
    optimizer = optimizer_spec["factory"](model.parameters())

    iteration_history = []
    loss_history = []
    sharpness_history = []

    for iteration in range(iterations + 1):
        with torch.no_grad():
            train_loss = loss_fn(model(x_train), y_train).item()

        iteration_history.append(iteration)
        loss_history.append(train_loss)
        sharpness_history.append(sharpness(model, x_train, y_train))

        if iteration == iterations:
            break

        x_batch, y_batch = select_batch(optimizer_spec["batch_mode"], batch_generator)
        optimizer.zero_grad()
        loss = loss_fn(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()

    return {
        "name": optimizer_spec["name"],
        "lr": optimizer_spec["lr"],
        "batch_mode": optimizer_spec["batch_mode"],
        "iterations": iteration_history,
        "losses": loss_history,
        "sharpness": sharpness_history,
        "final_loss": loss_history[-1],
        "final_sharpness": sharpness_history[-1],
    }


results = [run_experiment(optimizer_spec) for optimizer_spec in optimizer_specs]
[(result["name"], result["lr"], result["final_loss"], result["final_sharpness"]) for result in results]

[('GD', 0.1, 1.0469333733276187e-12, 18.918087005615234),
 ('SGD', 0.1, 6.142655828433874e-15, 18.914093017578125),
 ('SGD + Polyak momentum', 0.02, 1.471739397018723e-14, 18.574512481689453),
 ('Adam', 0.01, 3.234955706088449e-14, 21.339282989501953)]

In [6]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Train MSE", "Sharpness"),
)

colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]

for result, color in zip(results, colors):
    label = f"{result['name']} (lr={result['lr']})"
    fig.add_trace(
        go.Scatter(
            x=result["iterations"],
            y=result["losses"],
            mode="lines",
            name=label,
            legendgroup=result["name"],
            line={"color": color},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=result["iterations"],
            y=result["sharpness"],
            mode="lines",
            name=label,
            legendgroup=result["name"],
            showlegend=False,
            line={"color": color},
        ),
        row=2,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=[0, iterations],
        y=[2 / base_lr, 2 / base_lr],
        mode="lines",
        name="2 / base lr",
        line={"dash": "dash", "color": "black"},
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text="Iteration", row=2, col=1)
fig.update_yaxes(title_text="Train MSE", row=1, col=1)
fig.update_yaxes(title_text="Sharpness", row=2, col=1)
fig.update_layout(height=650, hovermode="x unified")

fig.show()